In [55]:
import pandas as pd
from pathlib import Path

def find_project_root() -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'pyproject.toml').exists():
            return parent
    raise FileNotFoundError('pyproject.toml is not in project root')

PROJECT_ROOT = find_project_root()
data_path = PROJECT_ROOT / 'data/raw' # data is loaded in this folder


In [56]:
feature_frame = pd.read_csv(f"{data_path}/feature_frame.csv")
ff = feature_frame

ff = ff.groupby("order_id").filter(lambda x: len(x) >= 5)

In [57]:
# --- Dates ---
ff["created_at"] = pd.to_datetime(ff["created_at"])
ff["order_date"] = pd.to_datetime(ff["order_date"])

# --- Discounts ---
# >1: values between 1.0 and 1.3, arithmetic error (final price > original price)
ff["discount_pct"] = ff["discount_pct"].clip(0, 1)

# --- Feature engineering ---
# Value 33 in days_since_purchase is a placeholder meaning "no purchase history"
ff["no_history_variant"] = (ff["days_since_purchase_variant_id"] == 33).astype(int)
ff["no_history_product_type"] = (ff["days_since_purchase_product_type"] == 33).astype(int)

# Household: binary flags, what matters is whether they have them or not
ff["has_children"] = (ff["count_children"] > 0).astype(int)
ff["has_babies"]   = (ff["count_babies"] > 0).astype(int)
ff["has_pets"]     = (ff["count_pets"] > 0).astype(int)

# Milestone 1: exploration phase

**Selecting The Prediction Target:**

Outcome represents whether a user actually purchased a given item in an order (1) or not (0). Since our goal is to predict which users are likely to buy a specific promoted item, it's the direct signal we want to learn from.

In [58]:
y = ff.outcome

In [59]:
ff['order_date'].describe()

count                          2880549
mean     2021-01-12 12:11:14.646395904
min                2020-10-05 00:00:00
25%                2020-12-16 00:00:00
50%                2021-01-22 00:00:00
75%                2021-02-14 00:00:00
max                2021-03-03 00:00:00
Name: order_date, dtype: object

In [60]:
# Definir features y target
feature_cols = [
    # Behavioural - most predictive according to EDA
    "ordered_before", "abandoned_before", "active_snoozed", "set_as_regular",
    # Product
    "normalised_price", "discount_pct", "global_popularity",
    # Household
    "count_adults", "people_ex_baby", "has_children", "has_babies", "has_pets",
    # Temporal
    "days_since_purchase_variant_id", "avg_days_to_buy_variant_id", "std_days_to_buy_variant_id",
    "days_since_purchase_product_type", "avg_days_to_buy_product_type", "std_days_to_buy_product_type",
    # Sentinel flags
    "no_history_variant", "no_history_product_type"
]

In [61]:
train_mask = ff["order_date"] < "2021-02-01"
val_mask   = (ff["order_date"] >= "2021-02-01") & (ff["order_date"] < "2021-03-01")
test_mask  = ff["order_date"] >= "2021-03-01"

X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f"Train:      {len(X_train):,} filas ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation: {len(X_val):,} filas ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test:       {len(X_test):,} filas ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nOutcome rate train: {y_train.mean():.3f}")
print(f"Outcome rate val:   {y_val.mean():.3f}")
print(f"Outcome rate test:  {y_test.mean():.3f}")

Train:      1,714,926 filas (59.5%)
Validation: 1,047,726 filas (36.4%)
Test:       117,897 filas (4.1%)

Outcome rate train: 0.013
Outcome rate val:   0.010
Outcome rate test:  0.008


In [62]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Evaluar en validación
val_preds = model.predict_proba(X_val)[:, 1]
pr_auc = average_precision_score(y_val, val_preds)
print(f"PR-AUC en validación: {pr_auc:.4f}")

PR-AUC en validación: 0.1657


In [63]:
train_preds = model.predict_proba(X_train)[:, 1]
pr_auc_train = average_precision_score(y_train, train_preds)
print(f"PR-AUC en train:      {pr_auc_train:.4f}")
print(f"PR-AUC en validación: {pr_auc:.4f}")

PR-AUC en train:      0.1634
PR-AUC en validación: 0.1657
